In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [2]:
import chars2vec

2023-10-20 04:39:24.052088: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-10-20 04:39:24.052168: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-10-20 04:39:24.054345: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-10-20 04:39:24.258381: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-20 04:39:25.535489: W tensorflow/compiler/

In [3]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)
tf.config.experimental.set_memory_growth(physical_devices[1], enable=True)

In [4]:
tqdm.pandas()

In [5]:
author_references = pd.read_hdf('AND_ORCID_ADS.h5')

In [6]:
author_references_exploded = author_references.explode(['author', 'aff']).explode('title')

In [7]:
author_references_exploded

,@path,family-name,given-names,title,abstract,author,aff
0,/0000-0003-1178-1001,Frenkel,Josif,Class of ghost-free non-Abelian gauge theories,We discuss a class of non-Abelian gauge theori...,"Frenkel, Josif","Instituto de Fisica, Universidade de São Paulo..."
1,/0000-0002-5215-6010,Bower,Richard,Clusters of Galaxies,Why do galaxies cluster? Are these clusters st...,"Materne, J.",Hamburg Observatory
2,/0000-0001-5974-7043,Branco,Gustavo C.,Weak nonleptonic decays of charmed hadrons in ...,We analyze the two-body weak nonleptonic decay...,"Branco, G.","Department of Physics, The City College of the..."
2,/0000-0001-5974-7043,Branco,Gustavo C.,Weak nonleptonic decays of charmed hadrons in ...,We analyze the two-body weak nonleptonic decay...,"Mohapatra, R. N.","Department of Physics, The City College of the..."
2,/0000-0001-5974-7043,Branco,Gustavo C.,Weak nonleptonic decays of charmed hadrons in ...,We analyze the two-body weak nonleptonic decay...,"Hagiwara, T.","The Rockefeller University, New York, New York..."
...,...,...,...,...,...,...,...
671132,/0000-0002-5518-499X,Dini,Daniele,Ability of a pore network model to predict flu...,The local flow field and seepage induced drag ...,"Sufian, Adnan",-
671132,/0000-0002-5518-499X,Dini,Daniele,Ability of a pore network model to predict flu...,The local flow field and seepage induced drag ...,"Knight, Chris",-
671132,/0000-0002-5518-499X,Dini,Daniele,Ability of a pore network model to predict flu...,The local flow field and seepage induced drag ...,"O'Sullivan, Catherine",-
671132,/0000-0002-5518-499X,Dini,Daniele,Ability of a pore network model to predict flu...,The local flow field and seepage induced drag ...,"van Wachem, Berend",-


In [8]:
# Function to extract the first letter of the first name and convert to lowercase
def extract_first_letter(name):
    if len(name.split(',')) > 1 and len(name.split(',')[1].strip()) > 1:
        return name.split(',')[1].strip()[0].lower()
    else:
        return ''

# Function to extract the last name and convert to lowercase
def extract_last_name(name):
    return name.split(',')[0].strip().lower()

# Filter rows based on the specified condition
filtered_df = author_references_exploded[author_references_exploded.progress_apply(lambda row: extract_first_letter(row['author']) == row['given-names'][0].lower() and extract_last_name(row['author']) == row['family-name'].lower(), axis=1)]

100%|██████████| 37084361/37084361 [04:15<00:00, 145183.09it/s]


In [13]:
name_dropped = filtered_df.drop(columns=['given-names', 'family-name'])

In [14]:
name_dropped.drop_duplicates(inplace = True)

In [15]:
name_dropped['author_first'] = name_dropped.apply(lambda x: extract_first_letter(x['author']).lower() , axis=1)

In [16]:
name_dropped['author_last'] = name_dropped.apply(lambda x: extract_last_name(x['author']).lower() , axis=1)

In [39]:
import pandas as pd
from itertools import combinations

# Function to apply to each column to flatten lists
def flatten_lists(column):
    return column.apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)

def process_group(group):
    data_list = []

    for i, j in combinations(group.itertuples(), 2):
        data = {
            '@path': [i[1]],
            'title': [i[2]],
            'abstract': [i[3]],
            'author': [i[4]],
            'aff': [i[5]],  # Added 'aff' after 'author'
            'author_first': [i[6]],
            'author_last': [i[7]],
            '@path2': [j[1]],
            'title2': [j[2]],
            'abstract2': [j[3]],
            'author2': [j[4]],
            'aff2': [j[5]],  # Added 'aff2' after 'author2'
            'author_first2': [j[6]],
            'author_last2': [j[7]],
            'label': [i[1] == j[1]]
        }
        data_list.append(data)

    if data_list:
        return pd.DataFrame(data_list).apply(flatten_lists)
    else:
        return pd.DataFrame()

group = name_dropped.groupby(['author_first', 'author_last'])
new_rows = group.progress_apply(process_group).reset_index(drop=True)

100%|██████████| 98483/98483 [18:43<00:00, 87.62it/s] 


In [40]:
new_rows

,@path,title,abstract,author,aff,author_first,author_last,@path2,title2,abstract2,author2,aff2,author_first2,author_last2,label
0,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,Empirical model of a bending IPMC actuator,We study ionomeric polymer-metal composite (IP...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,True
1,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,Finite element simulations of the bending of t...,This paper presents a electro-mechanical model...,"Aabloo, A.",Tartu Univ. (Estonia),a,aabloo,True
2,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,A distributed model of IPMC,This paper presents a distributed model of an ...,"Aabloo, A.",Tartu Univ. (Estonia),a,aabloo,True
3,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,A mechanical model of a non-uniform ionomeric ...,This paper describes a mechanical model of an ...,"Aabloo, Alvo","IMS Lab, Institute of Technology, Tartu Univer...",a,aabloo,True
4,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,A multilink manipulator with IPMC joints,IPMC (Ionic Polymer Metal Composite) is a clas...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12971061,/0000-0001-5194-2362,Spectral properties of quasi-one-dimensional c...,We determine the one-particle spectral functio...,"Bonačić Lošić, Ž.","Department of Physics, Faculty of Natural Scie...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmon of three-dimensional Dirac sem...,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True
12971062,/0000-0001-5194-2362,Photoemission properties of organic chain comp...,In this work we extend our previous considerat...,"Bonačić Lošić, Ž.","Department of Physics, Faculty Science, Univer...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmons in Weyl semimetals,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True
12971063,/0000-0001-5194-2362,Photoemission properties of organic chain comp...,In this work we extend our previous considerat...,"Bonačić Lošić, Ž.","Department of Physics, Faculty Science, Univer...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmon of three-dimensional Dirac sem...,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True
12971064,/0000-0001-5194-2362,Surface plasmons in Weyl semimetals,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmon of three-dimensional Dirac sem...,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True


In [41]:
new_rows.groupby('label').count()

,@path,title,abstract,author,aff,author_first,author_last,@path2,title2,abstract2,author2,aff2,author_first2,author_last2
label,,,,,,,,,,,,,,
False,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042,3150042
True,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024,9821024


In [42]:
new_rows.drop_duplicates(inplace = True)

In [43]:
new_rows.query('abstract == abstract2 & author == author2 & label == True')

,@path,title,abstract,author,aff,author_first,author_last,@path2,title2,abstract2,author2,aff2,author_first2,author_last2,label
10445,/0000-0003-2021-7728,On Holmboe's instability for smooth shear and ...,The linear stability of a stratified shear flo...,"Alexakis, Alexandros","National Center for Atmospheric Research, Boul...",a,alexakis,/0000-0003-2021-7728,On Holmboe's instability for smooth shear and ...,The linear stability of a stratified shear flo...,"Alexakis, Alexandros",-,a,alexakis,True
60252,/0000-0002-6988-3855,Quantum continuous measurements: The stochasti...,"The stochastic Schrödinger equation, of classi...","Barchielli, Alberto","Politecnico di Milano, Department of Mathemati...",a,barchielli,/0000-0002-6988-3855,Quantum continuous measurements: The stochasti...,"The stochastic Schrödinger equation, of classi...","Barchielli, Alberto",-,a,barchielli,True
80663,/0000-0002-1733-4408,Search for Supersymmetry at the LHC in Events ...,A search for events with jets and missing tran...,"Belyaev, A.","Moscow State University, Moscow, Russia",a,belyaev,/0000-0002-1733-4408,Search for Supersymmetry at the LHC in Events ...,A search for events with jets and missing tran...,"Belyaev, A.","Rutherford Appleton Laboratory, Didcot, United...",a,belyaev,True
80844,/0000-0002-1733-4408,Measurement of the weak mixing angle with the ...,A multivariate likelihood method to measure el...,"Belyaev, A.","Moscow State University, Moscow, Russia",a,belyaev,/0000-0002-1733-4408,Measurement of the weak mixing angle with the ...,A multivariate likelihood method to measure el...,"Belyaev, A.","Rutherford Appleton Laboratory, Didcot, United...",a,belyaev,True
81846,/0000-0002-1733-4408,Identification techniques for highly boosted W...,In searches for new physics in the energy regi...,"Belyaev, A.","Skobeltsyn Institute of Nuclear Physics, Lomon...",a,belyaev,/0000-0002-1733-4408,Identification techniques for highly boosted W...,In searches for new physics in the energy regi...,"Belyaev, A.",Rutherford Appleton Laboratory; School of Phys...,a,belyaev,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12927492,/0000-0002-4458-8550,Fast geometric gate operation of superconducti...,A scheme for coupling superconducting charge q...,"Xue, Zheng-Yuan",-,z,xue,/0000-0002-4458-8550,Fast geometric gate operation of superconducti...,A scheme for coupling superconducting charge q...,"Xue, Zheng-Yuan","Laboratory of Quantum Information Technology, ...",z,xue,True
12932998,/0000-0002-7186-2543,Goos-Hänchen-like shifts at a metal/supercondu...,"At a normal-metal/superconductor interface, an...","Yu, Zhi-Ming","Research Laboratory for Quantum Materials, Sin...",z,yu,/0000-0002-7186-2543,Goos-Hänchen-like shifts at a metal/supercondu...,"At a normal-metal/superconductor interface, an...","Yu, Zhi-Ming",-,z,yu,True
12933057,/0000-0002-3831-0579,Lithospheric Structure and Tectonic Processes ...,"Beneath ultraslow-spreading ridges, the oceani...","Yu, Zhiteng","Key Laboratory of Submarine Geosciences, Secon...",z,yu,/0000-0002-3831-0579,Lithospheric Structure and Tectonic Processes ...,"Beneath ultraslow-spreading ridges, the oceani...","Yu, Zhiteng","Second Institute of Oceanography, MNR, Hangzho...",z,yu,True
12944330,/0000-0003-1260-2079,Recursive solutions for Laplacian spectra and ...,The complete knowledge of Laplacian eigenvalue...,"Zhang, Zhongzhi",-,z,zhang,/0000-0003-1260-2079,Recursive solutions for Laplacian spectra and ...,The complete knowledge of Laplacian eigenvalue...,"Zhang, Zhongzhi","School of Computer Science, Fudan University, ...",z,zhang,True


In [44]:
same_orcid_drop_index = new_rows.query('abstract == abstract2 & author == author2 & label == True').index

In [45]:
len(filtered_df.drop_duplicates(subset = ['abstract', 'author'])), len(filtered_df)

(583311, 584895)

In [46]:
two_orcids_drop_index = new_rows.query('abstract == abstract2 & author == author2 & label == False').index

In [47]:
new_rows_updated = new_rows.drop(np.concatenate([two_orcids_drop_index, same_orcid_drop_index])).copy().reset_index(drop = True)

In [ ]:
#new_rows_updated.to_hdf('author_references_sept6th.h5', key = 'names')

In [49]:
new_rows_updated = pd.read_hdf('author_references_sept6th.h5')

In [50]:
new_rows_updated

,@path,title,abstract,author,aff,author_first,author_last,@path2,title2,abstract2,author2,aff2,author_first2,author_last2,label
0,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,Empirical model of a bending IPMC actuator,We study ionomeric polymer-metal composite (IP...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,True
1,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,Finite element simulations of the bending of t...,This paper presents a electro-mechanical model...,"Aabloo, A.",Tartu Univ. (Estonia),a,aabloo,True
2,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,A distributed model of IPMC,This paper presents a distributed model of an ...,"Aabloo, A.",Tartu Univ. (Estonia),a,aabloo,True
3,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,A mechanical model of a non-uniform ionomeric ...,This paper describes a mechanical model of an ...,"Aabloo, Alvo","IMS Lab, Institute of Technology, Tartu Univer...",a,aabloo,True
4,/0000-0002-0183-1282,Application of the Monte Carlo method for crea...,A program called mcgen was written for creatin...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,/0000-0002-0183-1282,A multilink manipulator with IPMC joints,IPMC (Ionic Polymer Metal Composite) is a clas...,"Aabloo, Alvo",Tartu Univ. (Estonia),a,aabloo,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12969924,/0000-0001-5194-2362,Spectral properties of quasi-one-dimensional c...,We determine the one-particle spectral functio...,"Bonačić Lošić, Ž.","Department of Physics, Faculty of Natural Scie...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmon of three-dimensional Dirac sem...,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True
12969925,/0000-0001-5194-2362,Photoemission properties of organic chain comp...,In this work we extend our previous considerat...,"Bonačić Lošić, Ž.","Department of Physics, Faculty Science, Univer...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmons in Weyl semimetals,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True
12969926,/0000-0001-5194-2362,Photoemission properties of organic chain comp...,In this work we extend our previous considerat...,"Bonačić Lošić, Ž.","Department of Physics, Faculty Science, Univer...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmon of three-dimensional Dirac sem...,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True
12969927,/0000-0001-5194-2362,Surface plasmons in Weyl semimetals,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,/0000-0001-5194-2362,Surface plasmon of three-dimensional Dirac sem...,The surface plasmon excitation spectrum is cal...,"Bonačić Lošić, Željana","University of Split, Faculty of Science, Ruder...",ž,bonačić lošić,True


In [51]:
import pandas as pd
#from sklearn.model_selection import train_test_split

# Calculate the desired sample size for each class
sample_size_per_class = new_rows_updated.groupby('label').count().query('label == False')['@path'].values[0]  # Half of the total sample size

# Split the DataFrame into two parts: one for each class
class_0_data = new_rows_updated[new_rows_updated['label'] == 0]
class_1_data = new_rows_updated[new_rows_updated['label'] == 1]

# Sample from each class with the desired sample size
sample_class_0 = class_0_data.sample(n=sample_size_per_class)
sample_class_1 = class_1_data.sample(n=sample_size_per_class)

# Concatenate the two samples to create a balanced dataset
balanced_sample = pd.concat([sample_class_0, sample_class_1])

# Shuffle the dataset to ensure randomness
balanced_sample = balanced_sample.sample(frac=1).reset_index(drop=True)

In [ ]:
balanced_sample.reset_index(drop = True).to_hdf('author_references_sept20th_full_balanced.h5', key = 'names')

In [ ]:
balanced_sample.sample(50000).reset_index(drop = True).to_hdf('author_references_Oct3rd_balanced.h5', key = 'names')

In [ ]:
balanced_sample.sample(10000).reset_index(drop = True)